In [1]:
from timescale import TimescaleDBManager
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import joblib
import json
import re
from functools import partial

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import IsolationForest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay,accuracy_score, precision_score, recall_score, f1_score,classification_report
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score, roc_curve, RocCurveDisplay
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler


from tensorflow.keras import layers, models
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense,BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from tensorflow.keras.models import load_model

from scikeras.wrappers import KerasRegressor
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import optuna
import tensorflow as tf
import seaborn as sns








In [2]:
load_dotenv()
user_env        = os.getenv("USER_DB")
pass_env        = os.getenv("PASS_DB")
host_env        = os.getenv("HOST_DB", "localhost")
port_env        = os.getenv("DB_PORT_EXPOSED", "5432")
db_env          = os.getenv("NAME_DB", "tfm_db")
FOLDER_SALIDA   = os.getenv("FOLDER_SALIDA")
FOLDER_TMP      = os.getenv("FOLDER_TMP")

SEED            = 100#int(os.getenv("SEED"))

db = TimescaleDBManager(
    user=user_env,
    password=pass_env,
    host=host_env,
    port=port_env,
    dbname=db_env
)
db.conectar()
conexion_postgresql = db.connection

#esto es para usar una conexion postgrews para pandas que da un warning indicando que solo esta testeado para sqlalchemy
engine = create_engine(f'postgresql+psycopg2://{user_env}:{pass_env}@{host_env}:{port_env}/{db_env}')

🔌 [OK] Conexión establecida con TimescaleDB.


In [3]:
SQL='select * from vectores'
df = pd.read_sql(SQL, engine)


In [4]:
lista_label = df['label'].unique().tolist()

In [5]:
len(lista_label)

54

In [6]:
df_result=df[ ['vector_id','execution_name','label'] ].copy()


In [7]:
df_result

,vector_id,execution_name,label
0,90244,light-oauth2-data-1719592986.tar,correct
1,90245,light-oauth2-data-1719592986.tar,correct
2,90246,light-oauth2-data-1719592986.tar,correct
3,90247,light-oauth2-data-1719592986.tar,correct
4,90333,light-oauth2-data-1719594842.tar,invalid_code_verifier_format_PKCE_400
...,...,...,...
90185,180429,light-oauth2-data-1725866154.tar,delete_user_404_no_user
90186,180430,light-oauth2-data-1725866154.tar,update_password_401_wrong_password
90187,180431,light-oauth2-data-1725866154.tar,update_password_404_user_not_found
90188,180432,light-oauth2-data-1725866154.tar,update_password_400_not_match


In [16]:
filename = f'{FOLDER_SALIDA}/lista_metric_name_vector.json'

with open(filename, "r", encoding="utf-8") as f:
    lista_metric_name_vector = json.load(f)

lista_limpia = [metrica.split('(')[0] for metrica in lista_metric_name_vector]

# 2. Contar los valores únicos totales
valores_unicos = set(lista_limpia)
N_metric_value = len(valores_unicos)

# <span style="color:#2ca02c">1. Recuperamos separación ya hecha en conjuntos de train, val y test</span>

In [9]:
SQL='select * from vectores_split'
df_split = pd.read_sql(SQL, engine)

In [10]:
df_result = pd.merge(df_result, df_split, on="vector_id", how="inner")

In [11]:
df_result

,vector_id,execution_name,label,split_type
0,90244,light-oauth2-data-1719592986.tar,correct,test
1,90245,light-oauth2-data-1719592986.tar,correct,train
2,90246,light-oauth2-data-1719592986.tar,correct,train
3,90247,light-oauth2-data-1719592986.tar,correct,test
4,90333,light-oauth2-data-1719594842.tar,invalid_code_verifier_format_PKCE_400,train
...,...,...,...,...
90185,180429,light-oauth2-data-1725866154.tar,delete_user_404_no_user,train
90186,180430,light-oauth2-data-1725866154.tar,update_password_401_wrong_password,val
90187,180431,light-oauth2-data-1725866154.tar,update_password_404_user_not_found,train
90188,180432,light-oauth2-data-1725866154.tar,update_password_400_not_match,val


In [12]:
dic_idx_train = {}
#dic_idx_resto = {}
dic_idx_val   = {}
dic_idx_test  = {}

In [13]:
for label in lista_label:
    idx_train = df_result[  (df_result['label']==label) & (df_result['split_type'] == 'train') ].index.values
    idx_val   = df_result[  (df_result['label']==label) & (df_result['split_type'] == 'val'  ) ].index.values
    idx_test  = df_result[  (df_result['label']==label) & (df_result['split_type'] == 'test') ].index.values
    dic_idx_train[label] = idx_train
    dic_idx_val[label]   = idx_val
    dic_idx_test[label]  = idx_test

<span style="color:#2ca02c"> Convetimos los vectores del df en una matriz, seran los datos que pasemos a los algoritmo   </span>


In [14]:
X_matrix = np.array(df['vector'].tolist())
X_matrix.shape

(90190, 354)

In [17]:
timesteps = 3

X_matrix_LSTM = X_matrix.reshape((X_matrix.shape[0], timesteps, N_metric_value))

In [18]:
X_matrix_LSTM.shape

(90190, 3, 118)

# <span style="color:#2ca02c">2. Preparacion de datos para entrenamiento/test</span>

In [19]:
idx_correct_train = dic_idx_train['correct']
idx_correct_val   = dic_idx_val['correct']
idx_correct_test  = dic_idx_test['correct']

In [20]:
N_datos_test_por_anomalia = int( len(idx_correct_test)  / (len(lista_label)-1) )

In [21]:
idx_test =idx_correct_test.copy()
for label in lista_label:
    if label=='correct':
        continue
    if len(dic_idx_test[label])<N_datos_test_por_anomalia:
        idx_test = np.concatenate((idx_test, dic_idx_test[label]))
    else:
        #idx_test.extend(dic_idx_test[label][:N_datos_test_por_anomalia])
        # Corrección: pon los dos arrays dentro de un paréntesis
        idx_test = np.concatenate((idx_test, dic_idx_test[label][:N_datos_test_por_anomalia]))
        
X_test  = X_matrix_LSTM[idx_test]

In [22]:
X_correct_train = X_matrix_LSTM[idx_correct_train]
X_correct_test  = X_matrix_LSTM[idx_correct_test]

In [23]:
df_result = pd.DataFrame({
    't_rel': df.loc[idx_test, 't_rel'],
    'execution_name': df.loc[idx_correct_test, 'execution_name'],
    'label': df.loc[idx_test, 'label']
})

In [24]:
df_result

,t_rel,execution_name,label
0,5.0,light-oauth2-data-1719592986.tar,correct
3,20.0,light-oauth2-data-1719592986.tar,correct
13,580.0,NaN,code_verifier_missing_pkce_400
20,595.0,NaN,verification_failed_pkce_400
24,565.0,NaN,code_verifier_too_long_pkce_400
...,...,...,...
90016,45.0,light-oauth2-data-1725858922.tar,correct
90126,5.0,light-oauth2-data-1725866154.tar,correct
90131,35.0,light-oauth2-data-1725866154.tar,correct
90133,45.0,light-oauth2-data-1725866154.tar,correct


# <span style="color:#2ca02c">3. Recuperación del LSTM autoencoder calculado</span>

In [26]:
paquete_joblib= joblib.load('result/models/AE_LSTM.joblib')

# 2. Recuperar las variables y el modelo de Keras usando su ruta
lstm_ae_opt = paquete_joblib['model']


# <span style="color:#2ca02c">4. Clasificacion de anomalias</span>

## <span style="color:#2ca02c">4.1. N/A</span>

## <span style="color:#2ca02c">4.2. Clasificador Random Forest usando espacio latente del LSTM Auto Encoder</span>

In [27]:
idx_train = np.array([], dtype=int)
idx_test  = np.array([], dtype=int)

In [28]:
for label in lista_label:
    if label=='correct':
        continue
    idx_train_tmp = dic_idx_train[label]
    idx_train     = np.concatenate((idx_train , idx_train_tmp))
    idx_test_tmp  = dic_idx_test[label]
    idx_test      = np.concatenate((idx_test  , idx_test_tmp ))
    

In [29]:
X_anomalia_train = X_matrix_LSTM[idx_train]
X_anomalia_test  = X_matrix_LSTM[idx_test]

In [30]:
X_anomalia_test.shape

(14647, 3, 118)

In [31]:
Y_train_etiquetas = df.loc[idx_train , 'label']
Y_test_etiquetas  = df.loc[idx_test  , 'label']

In [32]:
# 1. Creamos una entrada explícita con la forma correcta de tus datos (timesteps, features)
encoder_inputs = Input(shape=(timesteps, N_metric_value))

# 2. Reutilizamos exactamente la capa LSTM del encoder de tu modelo ya entrenado (lstm_ae_opt.layers[0])
encoder_lstm_layer = lstm_ae_opt.layers[0]
encoder_outputs = encoder_lstm_layer(encoder_inputs)

# 3. Construimos el modelo funcional del encoder limpio y sin errores
encoder_model = Model(inputs=encoder_inputs, outputs=encoder_outputs, name='functional_encoder')

In [33]:
# 4. Ahora ya puedes extraer las características latentes de forma segura y directa
X_train_latente = encoder_model.predict(X_anomalia_train , verbose=0)
X_test_latente  = encoder_model.predict(X_anomalia_test  , verbose=0)

In [34]:
X_train_latente.shape

(43862, 128)

In [35]:
clf_anomalias = RandomForestClassifier(random_state=SEED, n_estimators=100)
clf_anomalias.fit(X_train_latente, Y_train_etiquetas)

# 5. Predecir y evaluar
Y_pred_tipos = clf_anomalias.predict(X_test_latente)

print("=== REPORTE DE CLASIFICACIÓN (USANDO ESPACIO LATENTE) ===")
print(classification_report(Y_test_etiquetas, Y_pred_tipos))

=== REPORTE DE CLASIFICACIÓN (USANDO ESPACIO LATENTE) ===
                                               precision    recall  f1-score   support

           access_token_auth_header_error_401       0.00      0.00      0.00       290
          access_token_authorization_form_401       0.00      0.00      0.00       297
         access_token_client_id_not_found_404       0.02      0.01      0.01       307
         access_token_client_secret_wrong_401       0.02      0.01      0.01       287
             access_token_form_urlencoded_400       0.05      0.01      0.02       310
          access_token_illegal_grant_type_400       0.03      0.01      0.01       306
access_token_missing_authorization_header_400       0.02      0.01      0.01       299
     authorization_code_client_id_missing_400       0.01      0.00      0.01       282
     authorization_code_invalid_client_id_404       0.00      0.00      0.00       299
      authorization_code_invalid_password_401       0.00      0.00     

## <span style="color:#2ca02c">4.3. Clasificador Random Forest usando metricas originales +espacio latente del LSTM Auto Encoder</span>



In [38]:
print(X_anomalia_train.shape)
print(X_anomalia_test.shape)

(43862, 3, 118)
(14647, 3, 118)


In [39]:
n_samples_train = X_anomalia_train.shape[0]
X_anomalia_train_2d = X_anomalia_train.reshape(n_samples_train, -1)

n_samples_test = X_anomalia_test.shape[0]
X_anomalia_test_2d = X_anomalia_test.reshape(n_samples_test, -1)

In [40]:
# 2. Concatenar las features originales con las features latentes (eje de columnas)
X_train_enriquecido = np.hstack((X_anomalia_train_2d, X_train_latente))
X_test_enriquecido = np.hstack((X_anomalia_test_2d, X_test_latente))

# 3. Entrenar el RandomForest con este dataset enriquecido
clf_anomalias.fit(X_train_enriquecido, Y_train_etiquetas)
Y_pred_tipos = clf_anomalias.predict(X_test_enriquecido)

print(classification_report(Y_test_etiquetas, Y_pred_tipos))

                                               precision    recall  f1-score   support

           access_token_auth_header_error_401       0.46      0.53      0.49       290
          access_token_authorization_form_401       0.54      0.60      0.57       297
         access_token_client_id_not_found_404       0.57      0.62      0.59       307
         access_token_client_secret_wrong_401       0.51      0.45      0.48       287
             access_token_form_urlencoded_400       0.61      0.63      0.62       310
          access_token_illegal_grant_type_400       0.52      0.55      0.53       306
access_token_missing_authorization_header_400       0.55      0.50      0.52       299
     authorization_code_client_id_missing_400       0.58      0.54      0.56       282
     authorization_code_invalid_client_id_404       0.47      0.45      0.46       299
      authorization_code_invalid_password_401       0.59      0.70      0.64       320
 authorization_code_missing_response_type_

## <span style="color:#2ca02c">4.3. Clasificador Random Forest usando metricas originales +espacio latente del LSTM Auto Encoder</span>

In [39]:
# 2. Concatenar las features originales con las features latentes (eje de columnas)
X_train_enriquecido = np.hstack((X_anomalia_train, X_train_latente))
X_test_enriquecido = np.hstack((X_anomalia_test, X_test_latente))

# 3. Entrenar el RandomForest con este dataset enriquecido
clf_anomalias.fit(X_train_enriquecido, Y_train_etiquetas)
Y_pred_tipos = clf_anomalias.predict(X_test_enriquecido)

print(classification_report(Y_test_etiquetas, Y_pred_tipos))

                                               precision    recall  f1-score   support

           access_token_auth_header_error_401       0.51      0.53      0.52       290
          access_token_authorization_form_401       0.58      0.59      0.58       297
         access_token_client_id_not_found_404       0.58      0.62      0.60       307
         access_token_client_secret_wrong_401       0.58      0.52      0.55       287
             access_token_form_urlencoded_400       0.59      0.65      0.62       310
          access_token_illegal_grant_type_400       0.53      0.55      0.54       306
access_token_missing_authorization_header_400       0.55      0.49      0.52       299
     authorization_code_client_id_missing_400       0.65      0.63      0.64       282
     authorization_code_invalid_client_id_404       0.53      0.54      0.54       299
      authorization_code_invalid_password_401       0.58      0.71      0.64       320
 authorization_code_missing_response_type_

## <span style="color:#2ca02c">4.4. Clasificador Random Forest usando metricas originales + error de reconstruccion</span>

(43862, 3, 118)

In [43]:
# 1. Reconstruir los datos con el autoencoder completo
X_train_reconstruido = autoencoder_final.predict(X_anomalia_train)
X_test_reconstruido = autoencoder_final.predict(X_anomalia_test)

# 2. Calcular el error cuadrático por cada característica (cuánto se equivocó la red en cada columna)
error_train = (X_anomalia_train - X_train_reconstruido) ** 2
error_test = (X_anomalia_test - X_test_reconstruido) ** 2


error_train_2d = error_train.reshape(n_samples_train, -1)
error_test_2d = error_test.reshape(n_samples_test, -1)


# 3. Concatenar las características originales + los errores de reconstrucción
X_train_con_error = np.hstack((X_anomalia_train_2d, error_train_2d))
X_test_con_error = np.hstack((X_anomalia_test_2d, error_test_2d))


# 4. Entrenar el clasificador
clf_anomalias.fit(X_train_con_error, Y_train_etiquetas)
Y_pred_tipos = clf_anomalias.predict(X_test_con_error)

1371/1371 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
458/458 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [44]:
print(classification_report(Y_test_etiquetas, Y_pred_tipos))

                                               precision    recall  f1-score   support

           access_token_auth_header_error_401       0.42      0.47      0.45       290
          access_token_authorization_form_401       0.52      0.56      0.53       297
         access_token_client_id_not_found_404       0.53      0.56      0.54       307
         access_token_client_secret_wrong_401       0.50      0.45      0.47       287
             access_token_form_urlencoded_400       0.57      0.59      0.58       310
          access_token_illegal_grant_type_400       0.47      0.52      0.49       306
access_token_missing_authorization_header_400       0.46      0.42      0.44       299
     authorization_code_client_id_missing_400       0.54      0.49      0.51       282
     authorization_code_invalid_client_id_404       0.45      0.45      0.45       299
      authorization_code_invalid_password_401       0.55      0.63      0.59       320
 authorization_code_missing_response_type_